In [2]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from collections import Counter
from datetime import datetime, timezone
import sys

In [ ]:
ROOT = Path.cwd()
MODELS_DIR = ROOT.joinpath("../../models/thirdIterration/boost").resolve()
TRAIN_CSV = ROOT.joinpath("../../data/processed/train_df_clean.csv").resolve()
EVAL_CSV  = ROOT.joinpath("../../data/processed/eval_df_clean.csv").resolve()
OUT_DIR = ROOT.joinpath("../..data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLD = 0.5

FALLBACK_COLS_TO_DROP = ['Time', 'Attendance', 'Div']

def safe_colname(col: str) -> str:
    """Normalize column names to avoid collisions and problematic characters."""
    if not isinstance(col, str):
        col = str(col)
    return (col
            .replace('>', '_gt_')
            .replace('<', '_lt_')
            .replace('[', '_')
            .replace(']', '_')
            .replace('/', '_')
            .replace(' ', '_')
            .replace('%', 'pct')
            .replace('.', '_')
            )

def load_feature_list(models_dir: Path):
    p = next(models_dir.glob("feature_list_*.pkl"), None)
    if p is None:
        raise FileNotFoundError(f"No feature_list_*.pkl found in {models_dir}")
    feat = joblib.load(p)
    if not isinstance(feat, (list, tuple)):
        try:
            feat = list(feat)
        except Exception:
            raise RuntimeError("feature_list loaded but is not a list-like object")
    return list(feat), p

def find_model(models_dir: Path):
    lgb = next(models_dir.glob("*lightgbm*.pkl"), None)
    xgb = next(models_dir.glob("*xgboost*.pkl"), None)
    candidate = lgb or xgb
    if candidate is None:
        candidate = next(models_dir.glob("*.pkl"), None)
    if candidate is None:
        raise FileNotFoundError(f"No model .pkl found in {models_dir}")
    return candidate

def parse_dates_safe(df: pd.DataFrame, col='Date'):
    if col not in df.columns:
        raise RuntimeError(f"Date column '{col}' not present in dataframe")
    s = df[col].dropna().astype(str)
    if s.str.match(r'^\d{4}-\d{2}-\d{2}').sum() > len(s) * 0.6:
        return pd.to_datetime(df[col], errors='coerce', dayfirst=False)
    parsed = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
    mask = parsed.isna()
    if mask.any():
        parsed.loc[mask] = pd.to_datetime(df.loc[mask, col].astype(str), errors='coerce', dayfirst=False)
    return parsed

def dedupe_preserve_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def merge_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    dup = df.columns[df.columns.duplicated()].unique()
    if len(dup) == 0:
        return df
    df = df.T.groupby(level=0).mean().T
    return df

def main():
    print("Working dir:", ROOT)
    print("Looking for models in:", MODELS_DIR)
    if not MODELS_DIR.exists():
        raise FileNotFoundError(f"Models directory not found: {MODELS_DIR}")
    print("Train CSV:", TRAIN_CSV)
    print("Eval CSV :", EVAL_CSV)
    if not TRAIN_CSV.exists():
        raise FileNotFoundError(f"Train CSV not found: {TRAIN_CSV}")
    if not EVAL_CSV.exists():
        raise FileNotFoundError(f"Eval CSV not found: {EVAL_CSV}")

    feature_list_raw, feat_path = load_feature_list(MODELS_DIR)
    print("Loaded feature_list from:", feat_path, "len:", len(feature_list_raw))

    model_path = find_model(MODELS_DIR)
    model = joblib.load(model_path)
    print("Loaded model:", model_path.name)

    train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
    eval_df = pd.read_csv(EVAL_CSV, low_memory=False)

    train_df['Date'] = parse_dates_safe(train_df, 'Date')
    eval_df['Date']  = parse_dates_safe(eval_df, 'Date')
    print("train Date range:", train_df['Date'].min(), "->", train_df['Date'].max())
    print("eval  Date range:", eval_df['Date'].min(), "->", eval_df['Date'].max())

    train_end_date = pd.to_datetime(train_df['Date']).max()
    prospective = eval_df[pd.to_datetime(eval_df['Date']) > train_end_date].reset_index(drop=True)
    print("train_end_date:", train_end_date, "prospective rows:", prospective.shape[0])
    if prospective.shape[0] == 0:
        raise RuntimeError("No prospective rows found; check Date parsing and CSVs.")

    try:
        cols_to_drop 
    except NameError:
        cols_to_drop_local = FALLBACK_COLS_TO_DROP
    else:
        cols_to_drop_local = cols_to_drop

    X_eval = prospective.drop(columns=cols_to_drop_local, errors='ignore').copy()

    X_eval.columns = [safe_colname(c) for c in X_eval.columns]
    feature_list_norm = [safe_colname(f) for f in feature_list_raw]
    feature_list_norm = dedupe_preserve_order(feature_list_norm)

    X_eval = X_eval.apply(pd.to_numeric, errors='coerce')

    dup = X_eval.columns[X_eval.columns.duplicated()].unique()
    if len(dup):
        print("Merging duplicate columns found in X_eval:", list(dup))
        X_eval = merge_duplicate_columns(X_eval)
        print("After merge, X_eval.shape:", X_eval.shape)

    missing_feats = [f for f in feature_list_norm if f not in X_eval.columns]
    if missing_feats:
        print(f"{len(missing_feats)} features missing in prospective; filling with 0. Examples:", missing_feats[:10])
    X_eval = X_eval.reindex(columns=feature_list_norm, fill_value=0).fillna(0)

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_eval.values)[:, 1]
    else:
        preds_tmp = model.predict(X_eval.values)
        probs = preds_tmp.astype(float)

    preds = (probs >= THRESHOLD).astype(int)
    print("Predictions computed for", len(probs), "rows.")

    out = pd.DataFrame({
        "match_id": prospective.get("match_id", prospective.index.astype(str)),
        "date": pd.to_datetime(prospective["Date"]).dt.strftime("%Y-%m-%d"),
        "home": prospective.get("HomeTeam", ""),
        "away": prospective.get("AwayTeam", ""),
        "prob_home": probs,
        "pred_label": preds,
        "odds_B365H": prospective.get("B365H", np.nan),
        "actual": prospective.get("HomeWin", np.nan),
    })
    out["model_source"] = model_path.name
    out["snapshot_created_at"] = datetime.now(timezone.utc).isoformat()

    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out_path = OUT_DIR / f"predictions_snapshot_{ts}.csv"
    out.to_csv(out_path, index=False)
    print("Saved snapshot:", out_path)
    print(out.head().to_dict(orient='records'))

if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print("ERROR:", exc, file=sys.stderr)
        raise

Working dir: /Users/bashaar/Code/gitHubRep/Over-Under2.5Goals/notebooks/thirdIterration
Looking for models in: /Users/bashaar/Code/gitHubRep/Over-Under2.5Goals/models/thirdIterration/boost
Train CSV: /Users/bashaar/Code/gitHubRep/Over-Under2.5Goals/data/processed/train_df_clean.csv
Eval CSV : /Users/bashaar/Code/gitHubRep/Over-Under2.5Goals/data/processed/eval_df_clean.csv
Loaded feature_list from: /Users/bashaar/Code/gitHubRep/Over-Under2.5Goals/models/thirdIterration/boost/feature_list_20251109T124027Z.pkl len: 186


In [3]:
p = "../../data/processed/predictions_snapshot_20251109T143856Z.csv"
df = pd.read_csv(p)
print("rows:", len(df))
print("prob_home summary:")
print(df['prob_home'].describe())
print("pred_label counts:")
print(df['pred_label'].value_counts())

rows: 269
prob_home summary:
count    269.000000
mean       0.300161
std        0.109920
min        0.062966
25%        0.227199
50%        0.296475
75%        0.375829
max        0.609473
Name: prob_home, dtype: float64
pred_label counts:
pred_label
0    255
1     14
Name: count, dtype: int64


In [4]:
df['implied'] = 1.0 / df['odds_B365H'].replace(0, pd.NA)
df['edge'] = df['prob_home'] - df['implied']
df['exp_return_per_unit'] = df['prob_home'] * (df['odds_B365H'] - 1) - (1 - df['prob_home'])
df['ev_alt'] = df['odds_B365H'] * df['prob_home'] - 1.0

print("Examples of largest positive edges:")
print(df.sort_values('edge', ascending=False)[['date','home','away','prob_home','odds_B365H','implied','edge','exp_return_per_unit']].head(10).to_string(index=False))

pos = df[df['edge'] > 0]
total_ev = pos['exp_return_per_unit'].sum()
print("Number of positive-edge bets:", len(pos), "Total EV (sum of per-unit EV):", total_ev)

Examples of largest positive edges:
      date           home           away  prob_home  odds_B365H  implied     edge  exp_return_per_unit
2024-09-01     Man United      Liverpool   0.609473        3.70 0.270270 0.339203             1.255051
2025-02-26  Nott'm Forest        Arsenal   0.521147        3.90 0.256410 0.264737             1.032474
2025-01-18      Leicester         Fulham   0.444681        4.33 0.230947 0.213734             0.925470
2025-02-19    Aston Villa      Liverpool   0.455760        3.90 0.256410 0.199350             0.777465
2025-01-15      Leicester Crystal Palace   0.452126        3.90 0.256410 0.195715             0.763289
2025-01-04 Crystal Palace        Chelsea   0.455339        3.80 0.263158 0.192181             0.730289
2025-01-19        Everton      Tottenham   0.508583        2.80 0.357143 0.151440             0.424032
2025-01-16        Ipswich       Brighton   0.406160        3.90 0.256410 0.149749             0.584023
2025-01-25    Southampton      Newcas

In [5]:
top_picks = df.sort_values('exp_return_per_unit', ascending=False).head(50)
top_picks.to_csv("predictions_top_picks.csv", index=False)
print("Saved top picks to predictions_top_picks.csv")
top_picks[['date','home','away','prob_home','odds_B365H','implied','edge','exp_return_per_unit']].head(20)

Saved top picks to predictions_top_picks.csv


,date,home,away,prob_home,odds_B365H,implied,edge,exp_return_per_unit
9,2024-09-01,Man United,Liverpool,0.609473,3.70,0.270270,0.339203,1.255051
245,2025-02-26,Nott'm Forest,Arsenal,0.521147,3.90,0.256410,0.264737,1.032474
202,2025-01-25,Southampton,Newcastle,0.296475,6.50,0.153846,0.142629,0.927086
193,2025-01-18,Leicester,Fulham,0.444681,4.33,0.230947,0.213734,0.925470
230,2025-02-19,Aston Villa,Liverpool,0.455760,3.90,0.256410,0.199350,0.777465
184,2025-01-15,Leicester,Crystal Palace,0.452126,3.90,0.256410,0.195715,0.763289
172,2025-01-04,Crystal Palace,Chelsea,0.455339,3.80,0.263158,0.192181,0.730289
36,2024-09-28,Wolves,Liverpool,0.207986,8.00,0.125000,0.082986,0.663887
187,2025-01-16,Ipswich,Brighton,0.406160,3.90,0.256410,0.149749,0.584023
5,2024-08-31,West Ham,Man City,0.226122,7.00,0.142857,0.083264,0.582851
